# Natural Language Processing



This project will give you practical experience using Natural Language Processing techniques. This project is in three parts:
- in part 1) you will use a dataset in a CSV file
- in part 2) you will use the Wikipedia API to directly access content
on Wikipedia.
- in part 3) you will make your notebook interactive


### Part 1)



- The CSV file is available at https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv
- The file contains a list of famous people and a brief overview.
- The goal of part 1) is to ...
  1. Pick one person from the list ( the reference person ) and output 10 other people who's overview are "closest" to the reference person in a Natural Language Processing sense
  1. Also output the sentiment of the overview of the reference person



### Part 2)



- For the same reference person that you chose in Part 1), use the Wikipedia API to access the whole content of the reference person's Wikipedia page.
- The goal of Part 2) is to ...
  1. Print out the text of the Wikipedia article for the reference person
  1. Determine the sentiment of the text of the Wikipedia page for the reference person
  1. Collect the text of the Wikipedia pages from the 10 nearest neighbors from Part 1)
  1. Determine the nearness ranking of these 10 people to your reference person based on their entire Wikipedia page
  1. Compare, i.e. plot,  the nearest ranking from Step 1) with the Wikipedia page nearness ranking.  A difference of the rank is one means of comparison.



### Part 3)


Make an interactive notebook where a user can choose or enter a name and the notebook displays the 10 closest individuals.

In addition to presenting the project slides, at the end of the presentation each student will demonstrate their code using a famous person suggested by the other students that exists in the DBpedia set.


## Data Collection

### Import

In [24]:
import duckdb
import pandas as pd
import os
import numpy as np
from google.colab import userdata


# TextBlob: cleans text (singularize) and scores sentiment
from textblob import TextBlob, Word

# TfidfVectorizer: turns each overview into a row of numbers (one column per word)
from sklearn.feature_extraction.text import TfidfVectorizer

# NearestNeighbors: finds the rows (people) closest to a reference row
from sklearn.neighbors import NearestNeighbors



## Settings

In [25]:
# ===== Change these for each pass =====
n_rows = 1000        # Pass 05: 1000 | Pass 06: 8000 | Pass 07: 50000 (more than the 42,786 rows, so you get them all)
ref_name = "Jonathan Hoefler"
use_saved = False    # False = do the slow cleaning and save it to Hugging Face
                     # True  = load the cleaning you already saved (fast)

In [26]:
# Load the first n_rows rows (set in the Settings cell)
df = duckdb.query(f'select * from "{base_file}.parquet" limit {n_rows}').df()
df.shape

(1000, 3)

## Convert CSV to Parquet

In [27]:
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv'
url

'https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv'

In [28]:
# File name without the extension: 'NLP'
base_file = url.split("/")[-1].split(".")[0]
base_file

'NLP'

In [29]:
# Read the remote CSV and write it straight to a local parquet file (lesson 102's ETL step)
# No all_varchar here: DuckDB auto-detects types, so number columns stay numbers
con = duckdb.connect()
con.execute(f"""
    COPY (
        SELECT *
        FROM read_csv('{url}')
    )
    TO '{base_file}.parquet'
    (FORMAT PARQUET);
""")

! ls -l {base_file}.parquet

-rw-r--r-- 1 root root 49460246 Nov 27 17:38 NLP.parquet


### Push to Hugging Face

In [30]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account

'hmsealy'

In [31]:
hf_repo = os.environ["HF_REPO"] = "Data_Science-22"
hf_repo

'Data_Science-22'

In [32]:
hf_file = os.environ["HF_FILE"] = f"{base_file}.parquet"
hf_file

'NLP.parquet'

In [33]:
# Get the token from Colab Secrets. Only shows the first 3 and last 4 characters
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"

'hf_ ... kJTa'

In [34]:
! hf auth login --token ${HF_TOKEN}

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [35]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"

In [36]:
print(hf_upload.stdout)

✓ Uploaded
  url: https://huggingface.co/datasets/hmsealy/Data_Science-22/commit/53b83f43a5d0a777ceb6ff484e323f69b898e28f



### Download TextBlob word lists

In [37]:
%%capture
# %%capture hides the long download log
!python -m textblob.download_corpora

## Load Data

### Load first n_rows rows

In [38]:
# Pass 04 rule: use only the first 1000 rows of the CSV
df = duckdb.query(f'select * from "{base_file}.parquet" limit 1000').df()
df.shape

(1000, 3)

## Data checks

In [39]:
import requests                      # fetch web pages (HTML lecture, 315)
from urllib.parse import unquote     # turns %C3%A9 back into é

### Check 1: Missing names

In [40]:
# How many rows have no name at all? We want 0.
df["name"].isna().sum()

np.int64(0)

### Check 2: Duplicate names

In [41]:
# keep=False shows EVERY copy of a repeated name, not just the second one
df[df["name"].duplicated(keep=False)][["name"]]

,name


### Check 3: Names with parentheses

In [42]:
# Wikipedia adds (job) to tell apart people with the same name, e.g. "Paddy Dunne (Gaelic footballer)"
# regex=False means "(" is just a normal character here
df[df["name"].str.contains("(", regex=False)][["name"]]

,name
17,Paddy Dunne (Gaelic footballer)
27,Ian Mitchell (author)
31,Ted Hill (mathematician)
41,Bob Adams (American football)
50,Professor Lyrical (rapper)
...,...
972,Larry Sawyer (poet)
983,John Hart (speedway rider)
988,Mike Burton (rugby union)
989,Michael Rowe (director)


### Check 4: Names with commas

In [43]:
# Titles like "Anthony Gueterbock, 18th Baron Berkeley"
df[df["name"].str.contains(",", regex=False)][["name"]]

,name
13,"Anthony Gueterbock, 18th Baron Berkeley"
178,"John Taylor (cricketer, born 1937)"
200,"Mick O'Brien (footballer, born 1979)"
215,"Jose Luis Nazario, Jr."
334,"Chris Curran (footballer, born September 1971)"
526,"John James (footballer, born 1934)"
686,"Jesse Jackson, Jr."
732,"Kevin Rose (footballer, born 1960)"
825,"John O'Neill (footballer, born 1974)"
869,"Andrew Mawson, Baron Mawson"


### Check 5: Names with unusual characters

In [44]:
# Anything that is NOT a plain letter, space, period, apostrophe or hyphen.
# This catches digits, accents (é, ö), parentheses and commas.
df[df["name"].str.contains(r"[^A-Za-z .'\-]")][["name"]]

,name
13,"Anthony Gueterbock, 18th Baron Berkeley"
17,Paddy Dunne (Gaelic footballer)
27,Ian Mitchell (author)
31,Ted Hill (mathematician)
35,Freimut B%C3%B6rngen
...,...
983,John Hart (speedway rider)
985,"Peta Buscombe, Baroness Buscombe"
988,Mike Burton (rugby union)
989,Michael Rowe (director)


### Check 6: Name not found in its own overview

In [45]:
# Is the first word of the name in the overview text?
# If not, the person may be known by a stage name (e.g. G-Enka = Henry Kõrvits)
# or an accent was removed in the text (é -> e)
first_words = [n.lower().split()[0] for n in df["name"]]
df["name_in_text"] = [w in t for w, t in zip(first_words, df["text"])]

# ~ means "NOT", so this shows rows where the name was NOT found
df[~df["name_in_text"]][["name", "text"]]

,name,text
4,G-Enka,henry krvits born 30 december 1974 in tallinn ...
33,Bob Reece,robert scott reece born january 5 1951 is an a...
36,Th%C3%BCring Br%C3%A4m,thring brm born 10 april 1944 is a swiss compo...
72,Ulf-Dietrich Reips,prof dr ulfdietrich reips is a full professor ...
75,Vladimir Yurchenko,uladzimir vasilyevich yurchanka belarusian rus...
...,...,...
925,Tom Drury,thomas jay drury is an american writerhe was b...
943,H. Wayne Light,h wayne light phd born 1945 is an academic and...
960,Joseph Gitile Naituli,gitile naituli is a professor of management an...
966,Tony Ghelfi,anthony paul ghelfi born august 23 1961 is a f...


### Check 7: Page title from each URI

In [46]:
# <http://dbpedia.org/resource/Jonathan_Hoefler>  ->  Jonathan_Hoefler
df["title"] = (
    df["URI"]
    .str.replace("<http://dbpedia.org/resource/", "")
    .str.replace(">", "")
    .apply(unquote)
)

# Does the title match the name? (title uses _ instead of spaces)
df[df["title"].str.replace("_", " ") != df["name"]][["name", "title"]]

,name,title
35,Freimut B%C3%B6rngen,Freimut_Börngen
36,Th%C3%BCring Br%C3%A4m,Thüring_Bräm
73,Marcel J. Melan%C3%A7on,Marcel_J._Melançon
85,Zvonimir Juri%C4%87,Zvonimir_Jurić
122,Se%C3%A1n %C3%93g %C3%93 hAilp%C3%ADn,Seán_Óg_Ó_hAilpín
129,L%C3%A1zaro C%C3%A1rdenas Batel,Lázaro_Cárdenas_Batel
148,Ren%C3%A9 Froger,René_Froger
166,Ant%C3%B3nio Castanheira Neves,António_Castanheira_Neves
182,Mike P%C3%A9rez (baseball),Mike_Pérez_(baseball)
203,David Pastr%C5%88%C3%A1k,David_Pastrňák


### Fix coded names

In [47]:
# Some names are stored as web-address codes: "B%C3%B6rngen" should be "Börngen"
# unquote() turns them back into normal letters. Names without codes don't change.
df["name"] = df["name"].apply(unquote)

# Show the 4 fixed names
df.loc[[35, 36, 73, 85], "name"]

,name
35,Freimut Börngen
36,Thüring Bräm
73,Marcel J. Melançon
85,Zvonimir Jurić


### Summary of Wikipedia checks

In [48]:
# How many of each result?
df["wiki_status"].value_counts()

KeyError: 'wiki_status'

### People whose link is NOT ok

In [ ]:
df[df["wiki_status"] != "ok"][["name", "title", "wiki_status"]]

## Part 1

### Pick the reference person

In [49]:
ref_name = "Jonathan Hoefler"
ref_index = 12   # Jonathan's row number, found by looking at df (same as with 20 rows)
ref_index

12

### Clean text (singularize)

In [50]:
# Turn each word into its singular form so "typefaces" and "typeface" count as the same word
def singularize_text(text):
    words = text.split()
    singles = [Word(w).singularize() for w in words]
    return " ".join(singles)

# A different file for each pass size, e.g. NLP_clean_1000.parquet
clean_file = f"NLP_clean_{n_rows}.parquet"
clean_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/{clean_file}"

if use_saved:
    # Fast: load the cleaned text we saved last time (same way as the HF lecture)
    saved = duckdb.query(f'select * from "{clean_url}"').df()
    df["text_clean"] = saved["text_clean"]
else:
    # Slow: clean every overview, then save the result to a parquet file
    df["text_clean"] = df["text"].apply(singularize_text)
    df[["name", "text_clean"]].to_parquet(clean_file)

df[["name", "text_clean"]].head()

,name,text_clean
0,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,G-Enka,henry krvit born 30 december 1974 in tallinn b...


In [51]:
# stop_words='english' drops filler words like "the", "is", "and"
tfidf_vec = TfidfVectorizer(stop_words="english")

# fit_transform learns the vocabulary and builds the matrix:
# 100 rows (people) x one column per unique word
tf_idf = tfidf_vec.fit_transform(df["text_clean"])
tf_idf.shape

(1000, 30710)

### Teach NearestNeighbors

In [52]:
# Teach NearestNeighbors where all 100 people sit in "word space"
nn = NearestNeighbors().fit(tf_idf)

# Ask for 11 neighbors, not 10: the closest match is always Jonathan himself (distance 0)
distances, indices = nn.kneighbors(
    X = tf_idf[ref_index],
    n_neighbors = 11,
)
distances, indices

(array([[0.        , 1.01502089, 1.31063222, 1.33775524, 1.36145023,
         1.36269295, 1.36838708, 1.36868304, 1.36930606, 1.36940602,
         1.37292162]]),
 array([[ 12, 223, 163, 704, 925, 582, 778, 798, 800, 556, 643]]))

In [53]:
# Pull out the people at those row numbers, and add their distances
neighbors = df.iloc[indices[0]][["name"]].copy()
neighbors["distance"] = distances[0]

# Drop the first row (Jonathan himself) and keep only the 10 closest
neighbors = neighbors.iloc[1:11]

# Add a rank column: 1 = closest
neighbors.insert(0, "rank", range(1, 11))
neighbors.reset_index(drop=True)

,rank,name,distance
0,1,Tobias Frere-Jones,1.015021
1,2,Colleen McGill,1.310632
2,3,Jeff Madrick,1.337755
3,4,Tom Drury,1.361450
4,5,Robert H. Cumming,1.362693
5,6,Lemi Ghariokwu,1.368387
6,7,Laurence Gartel,1.368683
7,8,Brent Nicholson Earle,1.369306
8,9,Jean Valentine,1.369406
9,10,John Donald (jewellery designer),1.372922


### Sentiment

In [54]:
# Use the ORIGINAL text for sentiment, not the cleaned text
ref_blob = TextBlob(df.loc[ref_index, "text"])

# polarity: -1 (negative) to +1 (positive)
# subjectivity: 0 (factual) to 1 (opinion)
ref_blob.sentiment

Sentiment(polarity=0.19034090909090906, subjectivity=0.37683080808080804)

## Part 2

### Setup

In [55]:
%%capture
# The Wikipedia API package (same one used in lessons 504 and 506)
!pip3 install wikipedia-api

In [56]:
import re                      # regular expressions, for removing special characters
import wikipediaapi            # pulls page text from Wikipedia
import matplotlib.pyplot as plt

#unquote: turns web-address codes back into normal letters, e.g. "Beyonc%C3%A9" -> "Beyoncé"
# DBpedia links may store names this way; Wikipedia needs the real title to find the page
from urllib.parse import unquote


from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS   # the stop-word list TfidfVectorizer uses

### Get page titles

In [58]:
# Row numbers: Jonathan first, then his 15 nearest neighbors (10 + 5 spares)
rows = list(indices[0])

# Build a small table for Part 2
wiki_df = df.loc[rows, ["name", "URI"]].reset_index(drop=True)

# Part 1 rank: Jonathan = 0, closest neighbor = 1 ... last spare = 15
wiki_df["part1_rank"] = range(0, 16)

# The page title is the end of the DBpedia link:
# <http://dbpedia.org/resource/Jonathan_Hoefler>  ->  Jonathan_Hoefler
# unquote() turns codes like %C3%A9 back into letters like é
wiki_df["title"] = (
    wiki_df["URI"]
    .str.replace("<http://dbpedia.org/resource/", "")
    .str.replace(">", "")
    .apply(unquote)
)
wiki_df[["part1_rank", "name", "title"]]

ValueError: Length of values (16) does not match length of index (11)

### Collect Wikipedia text

In [ ]:
# Wikipedia asks every program to identify itself with a user_agent
wikip = wikipediaapi.Wikipedia(user_agent="neefmister")

def get_wiki_text(title):
    page = wikip.page(title)
    if page.exists():
        return page.text
    print(f"No Wikipedia page found for: {title}")   # warn us instead of crashing
    return

wiki_df["wiki_text"] = wiki_df["title"].apply(get_wiki_text)

# How many characters did we get for each person?
wiki_df["text_length"] = wiki_df["wiki_text"].str.len()
wiki_df[["name", "text_length"]]

### Drop people with no Wikipedia page

In [ ]:
# dropna removes any row where wiki_text is None (no Wikipedia page, e.g. Joerg Steineck)
wiki_df = wiki_df.dropna(subset=["wiki_text"])
wiki_df[["name", "part1_rank"]]

### Keep the 10 closest with a page

In [ ]:
# Rows are still in Part 1 order (closest first), so the first 11 rows
# are Jonathan + the 10 closest people who have a Wikipedia page.
# .copy() makes a fresh table, so pandas stops showing the SettingWithCopyWarning
wiki_df = wiki_df.head(11).reset_index(drop=True).copy()

# Save the original Part 1 rank, then renumber 0-10 so both rankings run 1-10
wiki_df["part1_rank_original"] = wiki_df["part1_rank"]
wiki_df["part1_rank"] = range(0, 11)
wiki_df[["name", "part1_rank_original", "part1_rank"]]

### Step 1: Reference person's article

In [ ]:
ref_wiki_text = wiki_df.loc[0, "wiki_text"]   # row 0 is Jonathan
print(ref_wiki_text)

### Step 2: Sentiment of the article

In [ ]:
# Use the ORIGINAL article text. Stemming would turn words like "outstanding"
# into "outstand", which TextBlob's sentiment word list doesn't recognize.
# polarity: -1 (negative) to +1 (positive)
# subjectivity: 0 (factual) to 1 (opinion)


TextBlob(ref_wiki_text).sentiment

### Clean text: case, special characters, stemming, lemmatization

In [ ]:
def clean_text(text, method):
    # 1. Change case: everything lowercase, so "Apple" and "apple" match
    text = text.lower()

    # 2. Remove possessives: "hoefler's" -> "hoefler"  (handles both ' and ’)
    text = re.sub(r"['’]s\b", "", text)

    # 3. Remove special characters: anything that is NOT a-z, 0-9 or a space becomes a space
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # 4. Split into words and drop stop words ("the", "was", "this" ...)
    #    We drop them BEFORE stemming. Otherwise "was" -> "wa" and "this" -> "thi"
    #    slip past the stop-word list, like the "wa" problem in Pass 01.
    words = [w for w in text.split() if w not in ENGLISH_STOP_WORDS]

    # 5. Stemming or lemmatization
    if method == "stem":
        words = [Word(w).stem() for w in words]        # chops endings: "designed" -> "design", "typefaces" -> "typefac"
    elif method == "lemma":
        words = [Word(w).lemmatize() for w in words]   # dictionary form: "typefaces" -> "typeface"

    return " ".join(words)

# See the difference on the first 300 characters of Jonathan's page
sample = ref_wiki_text[:300]
print("ORIGINAL:", sample, "\n")
print("STEM:    ", clean_text(sample, "stem"), "\n")
print("LEMMA:   ", clean_text(sample, "lemma"))

### Step 4: Wikipedia nearness ranking

In [ ]:
def wiki_ranking(method):
    # Clean all 11 pages the chosen way
    cleaned = wiki_df["wiki_text"].apply(lambda t: clean_text(t, method))

    # Same TF-IDF + nearest neighbors as Part 1, but on 11 full pages
    matrix = TfidfVectorizer().fit_transform(cleaned)
    nn_wiki = NearestNeighbors().fit(matrix)
    dist, idx = nn_wiki.kneighbors(matrix[0], n_neighbors=len(wiki_df))   # row 0 = Jonathan

    # idx lists rows from closest to farthest. Turn that into "row -> rank".
    ranks = pd.Series(range(len(idx[0])), index=idx[0])
    return ranks.sort_index().values   # back in wiki_df row order

wiki_df["wiki_rank_stem"]  = wiki_ranking("stem")
wiki_df["wiki_rank_lemma"] = wiki_ranking("lemma")

### Step 5: Compare rankings

In [ ]:
# Leave out Jonathan (row 0) and work out how far each person moved
compare = wiki_df.loc[1:, ["name", "part1_rank_original", "part1_rank", "wiki_rank_stem", "wiki_rank_lemma"]].copy()

# Positive = moved CLOSER on Wikipedia; negative = moved farther away
compare["change_stem"]  = compare["part1_rank"] - compare["wiki_rank_stem"]
compare["change_lemma"] = compare["part1_rank"] - compare["wiki_rank_lemma"]
compare

### Step 5, plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Each dot is one person: x = Part 1 rank, y = Wikipedia rank
ax.scatter(compare["part1_rank"], compare["wiki_rank_stem"],  label="Wikipedia (stemmed)",    s=80)
ax.scatter(compare["part1_rank"], compare["wiki_rank_lemma"], label="Wikipedia (lemmatized)", s=80, marker="x")

# Label the stemmed dots with names
for _, row in compare.iterrows():
    ax.annotate(row["name"], (row["part1_rank"], row["wiki_rank_stem"]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)

# Dashed line = "same rank both ways". Dots far from it changed the most.
ax.plot([1, 10], [1, 10], linestyle="--", color="gray", label="Same rank")

ax.set_xticks(range(1, 11))
ax.set_yticks(range(1, 11))
ax.set_xlabel("Part 1 rank (DBpedia overview)")
ax.set_ylabel("Part 2 rank (full Wikipedia page)")
ax.set_title("Nearness to Jonathan Hoefler: overview vs. full Wikipedia page")
ax.legend()
plt.show()

## Part 3

### Setup

In [ ]:
# interact turns a function into a widget (same import as the Interact lecture, 413)
from ipywidgets import interact

### List of names to choose from

In [ ]:
# All 20 names in A-Z order. Passing a list to interact makes a dropdown (lecture 413)
names = sorted(df["name"])
names

### Pick a person, see the 10 closest

In [ ]:
# @interact(person=names) puts a dropdown of names above the output.
# Every time you pick a new name, the function runs again.
@interact(person=names)
def show_closest(person="Jonathan Hoefler"):

    # Step 1: find the chosen person's row number
    match = df["name"] == person       # True for the chosen person, False for everyone else
    row = df.index[match][0]           # the row number where it's True

    # Step 2: same nearest-neighbors search as Part 1 (nn and tf_idf come from Part 1)
    # Ask for 11, because the closest match is always the person themselves
    dist_p, idx_p = nn.kneighbors(X=tf_idf[row], n_neighbors=11)

    # Step 3: print ranks 1-10, skipping position 0 (the person themselves)
    print(f"10 closest people to {person}:\n")
    for rank in range(1, 11):
        i = idx_p[0][rank]             # row number of this neighbor
        print(f"{rank:>2}. {df.loc[i, 'name']:<25} distance {dist_p[0][rank]:.3f}")